# LLM - Detect AI Generated Text: Data Loading & Exploration

This notebook is intended to be placed **in the same directory as the CSV files in this repository**.

It explores the official competition files and the additional training dataset:

- `train_essays.csv`
- `test_essays.csv`
- `train_prompts.csv`
- `sample_submission.csv`
- `Training_Essay_Data.csv`

The goal here is **data understanding only**: loading, checking schemas, inspecting labels and prompts, measuring essay lengths, finding missing/duplicate rows, and visualizing useful distributions before modeling.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)

DATA_DIR = Path.cwd()

print("Reading data from:")
print(DATA_DIR.resolve())


## 1. Locate the repository files

Keeping the notebook in the repository root means the CSV files can be loaded with simple relative paths.


In [ ]:
FILES = {
    "train": DATA_DIR / "train_essays.csv",
    "test": DATA_DIR / "test_essays.csv",
    "prompts": DATA_DIR / "train_prompts.csv",
    "submission": DATA_DIR / "sample_submission.csv",
    "external": DATA_DIR / "Training_Essay_Data.csv",
}

for name, path in FILES.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:10s} -> {status:7s} | {path.name}")


### Git LFS note

`Training_Essay_Data.csv` is tracked with Git LFS in this repository.  
If the file was cloned without downloading the actual LFS content, it may contain only a small Git LFS pointer instead of CSV data.

The helper below detects that case and gives a clear error.


In [ ]:
def check_not_lfs_pointer(path):
    if not path.exists():
        return

    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            first_line = f.readline().strip()
    except Exception:
        return

    if first_line.startswith("version https://git-lfs.github.com/spec/v1"):
        raise RuntimeError(
            f"{path.name} is currently a Git LFS pointer, not the real CSV file. "
            "Run `git lfs pull` in the repository and then rerun this notebook."
        )


if FILES["external"].exists():
    check_not_lfs_pointer(FILES["external"])


## 2. Load the datasets


In [ ]:
def load_csv(path, required=True):
    if not path.exists():
        if required:
            raise FileNotFoundError(
                f"{path.name} was not found in {DATA_DIR.resolve()}"
            )
        return None

    return pd.read_csv(path)


train = load_csv(FILES["train"])
test = load_csv(FILES["test"])
prompts = load_csv(FILES["prompts"], required=False)
sample_submission = load_csv(FILES["submission"], required=False)
external = load_csv(FILES["external"], required=False)

datasets = {
    "train_essays": train,
    "test_essays": test,
    "train_prompts": prompts,
    "sample_submission": sample_submission,
    "Training_Essay_Data": external,
}

print("Datasets loaded successfully.")


## 3. Dataset shapes and columns

The official training set normally contains the essay text together with the `generated` target:

- `generated = 0` → human-written
- `generated = 1` → AI-generated

The test set does not contain the target because that is what a model must predict.


In [ ]:
summary_rows = []

for name, df in datasets.items():
    if df is None:
        summary_rows.append({
            "dataset": name,
            "rows": np.nan,
            "columns": np.nan,
            "column_names": "Not available"
        })
    else:
        summary_rows.append({
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "column_names": ", ".join(df.columns)
        })

dataset_summary = pd.DataFrame(summary_rows)
dataset_summary


## 4. Preview each dataset


In [ ]:
print("TRAIN ESSAYS")
display(train.head())

print("\nTEST ESSAYS")
display(test.head())

if prompts is not None:
    print("\nTRAIN PROMPTS")
    display(prompts.head())

if sample_submission is not None:
    print("\nSAMPLE SUBMISSION")
    display(sample_submission.head())

if external is not None:
    print("\nEXTERNAL TRAINING DATA")
    display(external.head())


## 5. Data quality checks

We inspect:

- data types
- missing values
- exact duplicate rows
- duplicate essay text
- unique IDs


In [ ]:
def quality_report(df, name):
    print("=" * 80)
    print(name)
    print("=" * 80)

    print("\nShape:", df.shape)

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    missing = df.isna().sum()
    print(missing[missing > 0] if (missing > 0).any() else "No missing values")

    print("\nExact duplicate rows:", df.duplicated().sum())

    if "text" in df.columns:
        print("Duplicate text values:", df["text"].duplicated().sum())
        print("Empty text rows:", df["text"].fillna("").str.strip().eq("").sum())

    if "id" in df.columns:
        print("Unique IDs:", df["id"].nunique(), "/", len(df))


quality_report(train, "Official train_essays.csv")
quality_report(test, "Official test_essays.csv")

if external is not None:
    quality_report(external, "Training_Essay_Data.csv")


## 6. Target distribution

Class balance is important because the training data may contain very different numbers of human and AI-generated essays.


In [ ]:
def show_target_distribution(df, name):
    if "generated" not in df.columns:
        print(f"{name}: no `generated` column")
        return None

    counts = df["generated"].value_counts(dropna=False).sort_index()
    percentages = (
        df["generated"].value_counts(normalize=True, dropna=False)
        .sort_index()
        .mul(100)
        .round(2)
    )

    result = pd.DataFrame({
        "count": counts,
        "percentage": percentages
    })

    print(name)
    display(result)

    ax = counts.plot(kind="bar", figsize=(6, 4))
    ax.set_title(f"Target Distribution — {name}")
    ax.set_xlabel("generated")
    ax.set_ylabel("Number of essays")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()

    return result


train_target_distribution = show_target_distribution(train, "Official Train")

if external is not None:
    external_target_distribution = show_target_distribution(
        external, "Training_Essay_Data"
    )


## 7. Add simple text-length features

These features are useful for EDA and later can also become baseline model features.

No NLP library is required.


In [ ]:
def add_text_features(df):
    df = df.copy()

    text = df["text"].fillna("").astype(str)

    df["char_count"] = text.str.len()
    df["word_count"] = text.str.split().str.len()
    df["sentence_count"] = text.str.count(r"[.!?]+")
    df["avg_word_length"] = (
        text.str.replace(r"\s+", "", regex=True).str.len()
        / df["word_count"].replace(0, np.nan)
    )

    return df


train_eda = add_text_features(train)
test_eda = add_text_features(test)

external_eda = (
    add_text_features(external)
    if external is not None and "text" in external.columns
    else None
)

train_eda[["text", "char_count", "word_count", "sentence_count", "avg_word_length"]].head()


## 8. Essay-length statistics


In [ ]:
length_columns = [
    "char_count",
    "word_count",
    "sentence_count",
    "avg_word_length"
]

print("OFFICIAL TRAIN")
display(train_eda[length_columns].describe().round(2))

print("\nOFFICIAL TEST")
display(test_eda[length_columns].describe().round(2))

if external_eda is not None:
    print("\nEXTERNAL TRAINING DATA")
    display(external_eda[length_columns].describe().round(2))


## 9. Compare essay lengths by target


In [ ]:
if "generated" in train_eda.columns:
    train_length_by_class = (
        train_eda.groupby("generated")[length_columns]
        .agg(["mean", "median", "std", "min", "max"])
        .round(2)
    )

    display(train_length_by_class)

    human_words = train_eda.loc[
        train_eda["generated"] == 0, "word_count"
    ]

    ai_words = train_eda.loc[
        train_eda["generated"] == 1, "word_count"
    ]

    plt.figure(figsize=(9, 5))
    plt.hist(
        [human_words, ai_words],
        bins=30,
        alpha=0.7,
        label=["Human (0)", "AI-generated (1)"]
    )
    plt.title("Official Train — Essay Word Count by Target")
    plt.xlabel("Word count")
    plt.ylabel("Number of essays")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Prompt analysis

The official dataset contains `prompt_id`, while `train_prompts.csv` provides information about the writing prompts.

This section checks how essays are distributed across prompts and whether train/test prompt IDs overlap.


In [ ]:
if "prompt_id" in train.columns:
    print("TRAIN PROMPT COUNTS")
    display(
        train["prompt_id"]
        .value_counts()
        .rename_axis("prompt_id")
        .reset_index(name="essay_count")
    )

if "prompt_id" in test.columns:
    print("\nTEST PROMPT COUNTS")
    display(
        test["prompt_id"]
        .value_counts()
        .rename_axis("prompt_id")
        .reset_index(name="essay_count")
    )

if "prompt_id" in train.columns and "prompt_id" in test.columns:
    train_prompt_ids = set(train["prompt_id"].dropna().unique())
    test_prompt_ids = set(test["prompt_id"].dropna().unique())

    print("\nUnique train prompts:", len(train_prompt_ids))
    print("Unique test prompts: ", len(test_prompt_ids))
    print("Shared prompt IDs:   ", train_prompt_ids & test_prompt_ids)
    print("Train-only prompts:  ", train_prompt_ids - test_prompt_ids)
    print("Test-only prompts:   ", test_prompt_ids - train_prompt_ids)


## 11. Inspect `train_prompts.csv`


In [ ]:
if prompts is not None:
    print("Prompt columns:", list(prompts.columns))
    display(prompts)

    if "prompt_id" in prompts.columns and "prompt_id" in train.columns:
        prompt_counts = (
            train.groupby("prompt_id")
            .size()
            .reset_index(name="train_essay_count")
        )

        prompt_overview = prompts.merge(
            prompt_counts,
            on="prompt_id",
            how="left"
        )

        display(prompt_overview)
else:
    print("train_prompts.csv is not available.")


## 12. Inspect example essays

Reading a few examples from each target class is one of the fastest ways to understand the dataset.


In [ ]:
def print_examples(df, target_value, n=2, seed=42):
    if "generated" not in df.columns:
        return

    subset = df[df["generated"] == target_value]

    if len(subset) == 0:
        print(f"No rows found for generated={target_value}")
        return

    sample = subset.sample(min(n, len(subset)), random_state=seed)

    label_name = "AI-GENERATED" if target_value == 1 else "HUMAN"

    print("\n" + "=" * 100)
    print(label_name)
    print("=" * 100)

    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        print(f"\nExample {i}")
        if "id" in row.index:
            print("ID:", row["id"])
        if "prompt_id" in row.index:
            print("Prompt ID:", row["prompt_id"])

        text = str(row["text"])
        print(text[:1500])

        if len(text) > 1500:
            print("... [truncated]")


print_examples(train, target_value=0, n=2)
print_examples(train, target_value=1, n=2)


## 13. Train vs. test comparison

Large differences between train and test text lengths can indicate dataset shift.


In [ ]:
comparison = pd.DataFrame({
    "train_mean": train_eda[length_columns].mean(),
    "train_median": train_eda[length_columns].median(),
    "test_mean": test_eda[length_columns].mean(),
    "test_median": test_eda[length_columns].median(),
}).round(2)

comparison


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(
    train_eda["word_count"],
    bins=30,
    alpha=0.65,
    label="Train"
)
plt.hist(
    test_eda["word_count"],
    bins=30,
    alpha=0.65,
    label="Test"
)
plt.title("Train vs Test — Word Count Distribution")
plt.xlabel("Word count")
plt.ylabel("Number of essays")
plt.legend()
plt.tight_layout()
plt.show()


## 14. External training data vs. official training data

`Training_Essay_Data.csv` contains additional labeled essays with the columns:

- `text`
- `generated`

This section checks whether its distribution differs strongly from the official training set.


In [ ]:
if external_eda is not None:
    external_comparison = pd.DataFrame({
        "official_train_mean": train_eda[length_columns].mean(),
        "official_train_median": train_eda[length_columns].median(),
        "external_mean": external_eda[length_columns].mean(),
        "external_median": external_eda[length_columns].median(),
    }).round(2)

    display(external_comparison)

    plt.figure(figsize=(9, 5))
    plt.hist(
        train_eda["word_count"],
        bins=40,
        alpha=0.65,
        label="Official train"
    )
    plt.hist(
        external_eda["word_count"],
        bins=40,
        alpha=0.65,
        label="Training_Essay_Data"
    )
    plt.title("Official vs External Training Data — Word Count")
    plt.xlabel("Word count")
    plt.ylabel("Number of essays")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Training_Essay_Data.csv is not available.")


## 15. Check text overlap between datasets

Text overlap can cause data leakage or give misleading validation results if the same essay appears in multiple datasets.


In [ ]:
def normalized_text_series(df):
    return (
        df["text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )


train_text_norm = normalized_text_series(train)
test_text_norm = normalized_text_series(test)

train_test_overlap = set(train_text_norm) & set(test_text_norm)

print("Exact normalized text overlap: official train vs test =", len(train_test_overlap))

if external is not None and "text" in external.columns:
    external_text_norm = normalized_text_series(external)

    official_external_overlap = set(train_text_norm) & set(external_text_norm)
    external_test_overlap = set(external_text_norm) & set(test_text_norm)

    print(
        "Exact normalized text overlap: official train vs external =",
        len(official_external_overlap)
    )
    print(
        "Exact normalized text overlap: external vs test =",
        len(external_test_overlap)
    )


## 16. Sample submission check

For the competition, predictions are written into the `generated` column for each test `id`.


In [ ]:
if sample_submission is not None:
    display(sample_submission.head())

    print("Submission shape:", sample_submission.shape)
    print("Submission columns:", list(sample_submission.columns))

    if "id" in test.columns and "id" in sample_submission.columns:
        print(
            "Test IDs match submission IDs:",
            set(test["id"]) == set(sample_submission["id"])
        )
else:
    print("sample_submission.csv is not available.")


## 17. Final EDA summary


In [ ]:
print("Official train rows:", len(train))
print("Official test rows: ", len(test))

if "generated" in train.columns:
    print("\nOfficial target counts:")
    print(train["generated"].value_counts().sort_index())

if external is not None:
    print("\nExternal training rows:", len(external))

    if "generated" in external.columns:
        print("External target counts:")
        print(external["generated"].value_counts().sort_index())

print("\nMedian official train words:", round(train_eda["word_count"].median(), 2))
print("Median official test words: ", round(test_eda["word_count"].median(), 2))

if external_eda is not None:
    print("Median external words:      ", round(external_eda["word_count"].median(), 2))

print("\nEDA complete. The data is now ready for preprocessing and model experiments.")


## Next step

After this exploration notebook, a natural modeling path for this dataset is:

1. Clean and validate official + external training data
2. Build a TF-IDF baseline
3. Evaluate with stratified validation
4. Add stronger text features or transformer embeddings
5. Compare models using ROC-AUC
6. Produce `generated` probabilities for `test_essays.csv`

Keeping EDA separate from modeling makes the repository easier to understand and maintain.
